# Свой алгоритм SVD → метрики

Напишите функцию `my_svd(a, rank)` в ячейке ниже и запустите все ячейки (*Run All*).
Ноутбук прогонит её на стандартном наборе матриц, сравнит с эталонной SVD в FP64 и с
`torch.linalg.svd`, и покажет точность и время.

**Контракт функции**
- `a` — `torch.Tensor` формы `(m, n)` (float32/float64/complex), на устройстве `DEVICE`;
- `rank` — сколько сингулярных троек нужно (`rank == min(m, n)` для полной SVD);
- вернуть `(U, s, Vh)` формы `(m, k)`, `(k,)`, `(k, n)`, где `k >= rank` (лишнее обрежется);
  можно вернуть NumPy-массивы или только `s` (тогда проверяется только спектр).

Всё то же самое из терминала: `python -m algorithm_laboratory.scorecard --method mymodule:my_svd`.

In [ ]:
from pathlib import Path
import sys, importlib
import numpy as np
import torch

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "svdbench").is_dir():
        ROOT = candidate
        sys.path.insert(0, str(ROOT))
        break
else:
    raise RuntimeError("Откройте ноутбук из репозитория Pavel-s_SVD")

from algorithm_laboratory import scorecard
importlib.reload(scorecard)
from algorithm_laboratory.scorecard import evaluate, compare, table, summary, DEFAULT_SUITE
from svdbench.laboratory import MatrixCase

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
DEVICE

## 1. Ваш алгоритм

Пример ниже — односторонний Якоби (Хестенс) на чистом torch, чтобы было с чего начать.
Замените тело функции своим методом. `breakpoint()` внутри тоже работает.

In [ ]:
def my_svd(a, rank):
    """One-sided Jacobi: orthogonalize columns of W = A V by plane rotations."""
    m, n = a.shape
    transpose = m < n
    w = (a.mH if transpose else a).clone()
    v = torch.eye(w.shape[1], dtype=w.dtype, device=w.device)
    eps = torch.finfo(w.real.dtype if w.is_complex() else w.dtype).eps
    for sweep in range(30):
        rotated = False
        for p in range(w.shape[1] - 1):
            for q in range(p + 1, w.shape[1]):
                alpha = (w[:, p].abs() ** 2).sum().real
                beta = (w[:, q].abs() ** 2).sum().real
                gamma = (w[:, p].conj() @ w[:, q])
                if gamma.abs() <= eps * torch.sqrt(alpha * beta):
                    continue
                rotated = True
                phase = gamma / gamma.abs()
                zeta = (beta - alpha) / (2 * gamma.abs())
                t = torch.sign(zeta) / (zeta.abs() + torch.sqrt(1 + zeta ** 2))
                t = torch.where(zeta == 0, torch.ones_like(t), t)
                c = 1 / torch.sqrt(1 + t ** 2); s = c * t
                for x in (w, v):
                    xp, xq = x[:, p].clone(), x[:, q].clone()
                    x[:, p] = c * xp - s * phase.conj() * xq
                    x[:, q] = s * phase * xp + c * xq
        if not rotated:
            break
    sigma = torch.linalg.vector_norm(w, dim=0)
    order = torch.argsort(sigma, descending=True)
    sigma, w, v = sigma[order], w[:, order], v[:, order]
    u = w / torch.where(sigma > 0, sigma, torch.ones_like(sigma))
    u, sigma, vh = u[:, :rank], sigma[:rank], v[:, :rank].mH
    return (vh.mH, sigma, u.mH) if transpose else (u, sigma, vh)

## 2. Быстрая проверка на одной маленькой матрице

Чистый Python-Якоби медленный, поэтому сначала проверяем на крошечной матрице.
`MatrixCase.create(...)` принимает семейства `gaussian, low_rank, clustered, geometric, sparse, zero`;
`MatrixCase.load("matrix.npy")` — свою матрицу; `MatrixCase.from_spectrum([...], m=, n=)` — заданный спектр.

In [ ]:
small = [dict(name="gaussian_24x16", m=24, n=16, family="gaussian", dtype="float64"),
         dict(name="complex_16x24", m=16, n=24, family="gaussian", dtype="complex128"),
         dict(name="low_rank_k4", m=30, n=20, family="low_rank", rank=4, tail=1e-6, dtype="float64", output_rank=4)]
rows = evaluate(my_svd, small, device=DEVICE, repeat=3)
table(rows)

## 3. Сравнение со встроенными методами на стандартном наборе

Методы задаются функцией или именем: `"torch"`, `"numpy"`, `"mine"` (`my_algorithm.py`),
`"lab.randomized"`, `"lab.jacobi"`, и любой из 16 вариантов `svdbench.standalone`:
`bidiag_qr, bidiag_dc, jacobi, block_jacobi, qr, tsqr, gram_jacobi, gram_dc, polar_qdwh,
polar_newton_schulz, polar_halley, polar_polynomial, randomized, subspace, block_krylov, lanczos`.

Стандартный набор (`DEFAULT_SUITE`) — 8 матриц до 400×300. Для медленного Python-кода
уменьшите его или используйте `small`.

In [ ]:
print([c["name"] for c in DEFAULT_SUITE])
CASES = small                     # DEFAULT_SUITE, когда ваш метод достаточно быстрый
rows = compare({"my_svd": my_svd, "torch": "torch", "bidiag_qr": "bidiag_qr", "polar_qdwh": "polar_qdwh"},
               CASES, device=DEVICE, repeat=5)
df = table(rows)
df

In [ ]:
import pandas as pd
display(pd.DataFrame(summary(rows)))
df.pivot(index="case", columns="method", values="time_median_ms")

### Что означают столбцы

| столбец | смысл |
|---|---|
| `ok` | прошёл ли проверку: ортогональность U, V и невязка ≤ 100·eps·max(m,n); для усечённой SVD — невязка ≤ 1.05× оптимального хвоста (Эккарт–Янг) |
| `time_median_ms`, `time_mad_ms` | медиана и MAD времени вызова функции (без генерации матрицы и проверки) |
| `reconstruction_relative_fro` | ‖A − U Σ Vᴴ‖_F / ‖A‖_F |
| `residual_over_optimal_tail` | невязка / ‖σ_{k+1:}‖ — во сколько раз хуже лучшего ранга-k приближения |
| `resolved_spectrum_max_relative_error` | макс. относительная ошибка σᵢ (для σ выше уровня шума FP64) |
| `spectrum_error_over_sigma1` | макс. \|σᵢ − σᵢ_ref\| / σ₁ |
| `orthogonality_u/v_...` | ‖UᴴU − I‖_F / √k и то же для V |
| `left/right_triplet_relative_fro` | ‖A vᵢ − σᵢ uᵢ‖ и ‖Aᴴ uᵢ − σᵢ vᵢ‖ (относительно ‖A‖) |
| `right_subspace_leakage_...` | насколько правое подпространство вышло за эталонное (только если есть спектральная щель) |
| `singular_values_nonnegative_sorted` | σ ≥ 0 и отсортированы по убыванию |

## 4. Свои матрицы и графики

In [ ]:
import matplotlib.pyplot as plt
mine = [r for r in rows if r["method"] == "my_svd" and r["status"] == "done"]
ref = {r["case"]: r for r in rows if r["method"] == "torch"}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
names = [r["case"] for r in mine]
ax[0].bar(names, [r["reconstruction_relative_fro"] for r in mine], label="my_svd")
ax[0].scatter(names, [ref[c]["reconstruction_relative_fro"] for c in names], color="k", zorder=3, label="torch")
ax[0].set_yscale("log"); ax[0].set_title("‖A − UΣVᴴ‖ / ‖A‖"); ax[0].legend()
ax[1].bar(names, [r["time_median_ms"] for r in mine], label="my_svd")
ax[1].scatter(names, [ref[c]["time_median_ms"] for c in names], color="k", zorder=3, label="torch")
ax[1].set_yscale("log"); ax[1].set_title("время, мс (медиана)"); ax[1].legend()
for x in ax: x.tick_params(axis="x", rotation=30)
plt.tight_layout()

In [ ]:
# Своя матрица: заданный спектр с кластером, или .npy из бенчмарка
custom = [MatrixCase.from_spectrum([5, 5, 1, 1e-3, 1e-9, 0], m=10, n=8, seed=1)]
# custom = [MatrixCase.load("experiments/broad-20261012/lowrank2048x1024_k32/matrix.npy")]
table(evaluate(my_svd, custom, device=DEVICE, repeat=3))

In [ ]:
# Сохранить результаты
scorecard.save(rows, ROOT / "laboratory_runs" / "scorecard.csv")

## Дальше

- Пошаговая трассировка внутренних итераций (снимки Q, B, ошибки на каждой итерации):
  `python -m algorithm_laboratory.experiment --algorithm mine`, см. `laboratory.ipynb`.
- GPU на Hetzner: тот же ноутбук/CLI, `DEVICE = "cuda:0"`, Python `/root/svdbench/.venv/bin/python`
  (см. `docs/ALGORITHM_LABORATORY.md`).